# Machine Downtime Analysis — Part 1: Your First SQL Queries

**The scenario:** You're the new operations analyst at a manufacturing plant. The plant manager hands you a log of every time a machine went down from June through August 2026 and asks:

> *"Where are we losing the most production time, and why?"*

That's a real analyst question. In this notebook you'll load the log into a SQL database and start answering it.

**Before you start:** put `downtime_log.csv` in the **same folder** as this notebook.

**How to run a cell:** click in it and press **Shift + Enter**. Run the cells in order, top to bottom.

## Step 1: Load the tools

- **pandas** — you know this from Codecademy. It reads the CSV file.
- **sqlite3** — a small SQL database that comes built into Python. Nothing to install.

In [3]:
%pip install pandas

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 9.6/9.6 MB 59.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 12.6/12.6 MB 80.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Acer\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [4]:
# pandas: reads files and shows data as neat tables
import pandas as pd

# sqlite3: a lightweight SQL database built into Python (no install needed)
import sqlite3

## Step 2: Load the CSV into a SQL database

Real analysts usually pull data from a database, not a CSV. So we'll put our CSV **into** a database first, then practice querying it with SQL — the way you would on the job.

In [5]:
# Read the CSV file into a pandas DataFrame (a table in memory)
df = pd.read_csv("downtime_log.csv")

# Create a database. ":memory:" means it lives in memory only,
# so it disappears when you close the notebook. That's fine for practice,
# since this cell rebuilds it every time you run it.
conn = sqlite3.connect(":memory:")

# Copy the DataFrame into the database as a table named "downtime".
#   index=False       -> don't add pandas' row numbers as an extra column
#   if_exists="replace" -> if the table already exists, overwrite it
df.to_sql("downtime", conn, index=False, if_exists="replace")

print("Loaded", len(df), "downtime events into the 'downtime' table.")

Loaded 621 downtime events into the 'downtime' table.


## Step 3: A helper so every query shows as a nice table

Instead of typing the same line over and over, we write a tiny function. You give it a SQL query (as text), it runs it against our database and returns a table.

In [6]:
def sql(query):
    """Run a SQL query on our database and return the result as a table."""
    # pd.read_sql_query sends the query to the database (conn)
    # and turns the answer into a pandas DataFrame, which displays nicely
    return pd.read_sql_query(query, conn)

## Step 4: Look at the data

**Always look at your data before analyzing it.** Here's what each column means:

| Column | Meaning |
|---|---|
| `event_id` | A unique number for each downtime event |
| `machine` | Which machine went down |
| `line` | Which production line/area the machine is in |
| `start_time` | When it went down |
| `shift` | Which shift was running (1st, 2nd, 3rd) |
| `cause` | Why it went down |
| `duration_min` | How many minutes it was down |

### Your first SQL query
- `SELECT *` → give me **all** columns (`*` means "everything")
- `FROM downtime` → from the table named downtime
- `LIMIT 10` → only show the first 10 rows (the full table has 600+)

SQL keywords are written in CAPITALS by tradition. SQL doesn't require it, but it makes queries easier to read.

In [7]:
sql("""
    SELECT *
    FROM downtime
    LIMIT 10
""")

,event_id,machine,line,start_time,shift,cause,duration_min
0,1,Press 2,Stamping,2026-06-01 07:02,1st,Pneumatic leak,33
1,2,Press 2,Stamping,2026-06-01 20:28,2nd,Jam,23
2,3,CNC 2,Machining,2026-06-01 03:17,3rd,Jam,9
3,4,CNC 1,Machining,2026-06-01 23:22,3rd,Sensor failure,37
4,5,CNC 2,Machining,2026-06-01 23:35,3rd,Preventive maintenance,78
5,6,Conveyor A,Assembly,2026-06-01 17:05,2nd,Material shortage,49
6,7,Press 2,Stamping,2026-06-01 19:13,2nd,Preventive maintenance,102
7,8,CNC 2,Machining,2026-06-01 01:10,3rd,Preventive maintenance,89
8,9,Welder 1,Assembly,2026-06-01 09:14,1st,Operator error,6
9,10,Welder 1,Assembly,2026-06-02 11:25,1st,Jam,25


## Step 5: Filter rows with `WHERE`

`WHERE` keeps only the rows that match a condition. It works like an `if` statement in Python.

**Question:** Which events lasted longer than 2 hours (120 minutes)?

- `SELECT machine, cause, duration_min` → only show these 3 columns
- `WHERE duration_min > 120` → only rows where the machine was down over 120 min
- `ORDER BY duration_min DESC` → sort by duration, biggest first (`DESC` = descending; `ASC` = smallest first)

In [8]:
sql("""
    SELECT machine, cause, duration_min
    FROM downtime
    WHERE duration_min > 120
    ORDER BY duration_min DESC
""")

,machine,cause,duration_min
0,Press 1,Electrical fault,177
1,Press 2,Electrical fault,172
2,Conveyor A,Electrical fault,169
3,Press 2,Electrical fault,157
4,Welder 1,Electrical fault,156
5,CNC 1,Electrical fault,143
6,Press 3,Electrical fault,142
7,Welder 1,Electrical fault,141
8,Welder 1,Electrical fault,137
9,CNC 2,Electrical fault,131


## Step 6: Summarize with `GROUP BY` — the most important SQL skill

This is where SQL gets powerful, and it's what analysts do all day.

`GROUP BY machine` puts all rows for the same machine into one pile, then collapses each pile into **one row**. You then tell SQL how to summarize each pile:

- `COUNT(*)` → how many rows (events) are in the pile
- `SUM(duration_min)` → add up all the minutes in the pile
- `AS total_minutes` → give that result a readable column name

It's the same idea as `df.groupby("machine")` in pandas.

**Question:** Which machines lose the most total time?

In [9]:
sql("""
    SELECT machine,
           COUNT(*)          AS events,         -- how many times it went down
           SUM(duration_min) AS total_minutes   -- total minutes lost
    FROM downtime
    GROUP BY machine
    ORDER BY total_minutes DESC
""")

,machine,events,total_minutes
0,Press 2,80,4041
1,Conveyor A,74,3356
2,Welder 1,77,3321
3,CNC 1,87,3229
4,CNC 2,82,3158
5,Press 3,74,2850
6,Packager,87,2812
7,Press 1,60,2550


**What to notice:** `--` starts a comment in SQL, just like `#` in Python.

Look at the top of that list. Is one machine clearly worse than the others? Hold that thought.

## Step 7: Dig deeper — *why* is the worst machine going down?

Finding the problem machine is step one. The plant manager's next question will be **"why?"**

Here we combine `WHERE` (only look at one machine) with `GROUP BY` (summarize by cause).

`ROUND(AVG(duration_min), 1)` → the **average** minutes per event, rounded to 1 decimal place.

👉 **If a different machine topped your list in Step 6**, change `'Press 2'` below to that machine's name. Note that text values in SQL go inside **single quotes**.

In [10]:
sql("""
    SELECT cause,
           COUNT(*)                     AS events,
           SUM(duration_min)            AS total_minutes,
           ROUND(AVG(duration_min), 1)  AS avg_minutes
    FROM downtime
    WHERE machine = 'Press 2'
    GROUP BY cause
    ORDER BY total_minutes DESC
""")

,cause,events,total_minutes,avg_minutes
0,Pneumatic leak,26,1887,72.6
1,Preventive maintenance,6,528,88.0
2,Electrical fault,4,477,119.3
3,Material shortage,6,295,49.2
4,Tool change,12,253,21.1
5,Sensor failure,4,208,52.0
6,Jam,12,200,16.7
7,Operator error,10,193,19.3


## 🔧 Your turn

You have all the tools now: `SELECT`, `WHERE`, `GROUP BY`, `COUNT`, `SUM`, `ORDER BY`.

**Challenge 1:** Which **cause** loses the most total minutes across the whole plant? *(Hint: copy the Step 6 query and change which column you group by.)*

**Challenge 2:** Group by **shift** and count how many events had the cause `'Operator error'`. Does one shift stand out? *(Hint: you'll need both `WHERE` and `GROUP BY`.)*

Write each query in the empty cells below. With your troubleshooting background, think about what you'd tell the plant manager based on what you find.

In [14]:
# Challenge 1: total minutes lost by 
sql("""
    SELECT cause,
        COUNT(*) AS events,                -- total minutes lost
        SUM(duration_min) AS total_minutes -- how many total minutes
    FROM downtime
    GROUP BY cause
    ORDER BY total_minutes DESC
""")    

,cause,events,total_minutes
0,Preventive maintenance,62,5541
1,Pneumatic leak,67,5125
2,Electrical fault,34,3663
3,Material shortage,55,3038
4,Jam,151,2322
5,Tool change,112,2239
6,Sensor failure,51,1820
7,Operator error,89,1569


In [26]:
# Challenge 2: operator errors by shift
sql("""
    SELECT shift,
        COUNT(*) AS events,               -- total operator error event
        SUM(duration_min) AS total_mins   -- show many total minutes
    FROM downtime
    WHERE cause = 'Operator error'
    GROUP BY shift
    ORDER BY total_mins DESC
""")

,shift,events,total_mins
0,3rd,46,847
1,2nd,24,373
2,1st,19,349


In [27]:
sql("""
    SELECT shift,
           COUNT(*) AS all_events,       -- every downtime event on this shift
           SUM(CASE WHEN cause = 'Operator error' THEN 1 ELSE 0 END)
                    AS operator_errors,  -- count only the operator errors
           ROUND(100.0 * SUM(CASE WHEN cause = 'Operator error' THEN 1 ELSE 0 END)
                 / COUNT(*), 1) AS pct_operator_error  -- errors as a % of all events
    FROM downtime
    GROUP BY shift
    ORDER BY pct_operator_error DESC
""")

,shift,all_events,operator_errors,pct_operator_error
0,3rd,182,46,25.3
1,2nd,207,24,11.6
2,1st,232,19,8.2
